In [ ]:
!pip install ultralytics pandas ipython matplotlib opencv-python

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XpMwhGeTQsdlI8UuG4OG")
project = rf.workspace("shah-xxxqs").project("violence-3h8pw")
version = project.version(1)
dataset = version.download("yolo26")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Violence-1 in yolo26:: 100%|██████████| 5680/5680 [00:00<00:00, 9589.94it/s]


In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data={dataset.location}/data.yaml epochs=35 imgsz=832 verbose=True patience=10 lr0=0.001 optimizer=SGD device=0

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Violence-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=10, perspective=0.

In [ ]:
import yaml
import os

# 1. Setup the directory path from your screenshot
dataset_path = '/content/violence-detection-1/'
class_names = ['violence', 'non_violence']

# 2. Create the configuration dictionary
data_config = {
    'train': os.path.join(dataset_path, 'train/images'),
    'val': os.path.join(dataset_path, 'valid/images'),
    'test': os.path.join(dataset_path, 'test/images'),
    'nc': len(class_names),
    'names': class_names
}

# 3. Save the data.yaml file in the correct location
yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"✅ Success: data.yaml created at {yaml_path}")

In [ ]:
from ultralytics import YOLO
import shutil
import os
from google.colab import drive

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# Define Project Paths
PROJECT_NAME = 'School_Violence_Project'
RUN_NAME = 'v1_fast_train_10k'
DRIVE_BACKUP_PATH = f'/content/drive/MyDrive/thesis/{RUN_NAME}/'



base_path = '/content/violence-detection-1'
splits = ['train', 'valid', 'test']

for split in splits:
    split_path = os.path.join(base_path, split)

    # 1. Create the standard YOLO folders
    os.makedirs(os.path.join(split_path, 'images'), exist_ok=True)
    os.makedirs(os.path.join(split_path, 'labels'), exist_ok=True)

    # 2. Move everything from subfolders (violence/non_violence) to the new images/labels
    # Note: This assumes your images and labels are currently mixed or in these subfolders
    for sub in ['violence', 'non_violence']:
        sub_path = os.path.join(split_path, sub)
        if os.path.exists(sub_path):
            for file in os.listdir(sub_path):
                file_path = os.path.join(sub_path, file)
                # If it's an image, move to images folder
                if file.endswith(('.jpg', '.jpeg', '.png')):
                    shutil.move(file_path, os.path.join(split_path, 'images', file))
                # If it's a label (.txt), move to labels folder
                elif file.endswith('.txt'):
                    shutil.move(file_path, os.path.join(split_path, 'labels', file))

print("✅ Folders reorganized for YOLOv8!")

In [ ]:
# ==========================================
#  POST-TRAINING BACKUP FUNCTION
# ==========================================
def backup_results_to_drive():
    """Copies weights and evaluation graphs to Google Drive."""
    source_dir = f'{PROJECT_NAME}/{RUN_NAME}/'

    # Create the backup directory if it doesn't exist
    if not os.path.exists(DRIVE_BACKUP_PATH):
        os.makedirs(DRIVE_BACKUP_PATH)

    # List of essential files for the Thesis/Research
    essential_files = [
        'weights/best.pt',
        'results.png',
        'confusion_matrix.png',
        'F1_curve.png',
        'P_curve.png',
        'R_curve.png',
        'results.csv'
    ]

    print(f"--- Starting Backup to: {DRIVE_BACKUP_PATH} ---")
    for file_path in essential_files:
        src = os.path.join(source_dir, file_path)
        dst = os.path.join(DRIVE_BACKUP_PATH, os.path.basename(file_path))

        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"✅ Backed up: {os.path.basename(file_path)}")
        else:
            print(f"⚠️ Warning: {file_path} not found.")

# Run the backup after training is complete
backup_results_to_drive()

In [ ]:
import cv2
from ultralytics import YOLO
from google.colab import files

# Load trained model
model = YOLO("best.pt")

In [ ]:
def detect_on_video(input_path, output_path="output_detected.mp4"):

    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print("Error opening video file")
        return

    # video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # create output video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print("Running YOLO detection...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # inference
        results = model(frame)

        # draw bounding boxes
        annotated_frame = results[0].plot()

        # write frame
        out.write(annotated_frame)

    cap.release()
    out.release()

    print("Detection finished")
    print("Saved video as:", output_path)

In [ ]:
detect_on_video("/content/file_002003.mp4")

In [ ]:
files.download("output_detected.mp4")